Otestovanie prístupu k bezplatnej GPU od Colabu:

In [ ]:
!nvidia-smi

Sun Oct  4 10:03:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Inštalácia AI modelu TripoSr:

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR

Cloning into 'TripoSR'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 164 (delta 66), reused 45 (delta 45), pack-reused 66 (from 1)
Receiving objects: 100% (164/164), 36.71 MiB | 25.64 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/TripoSR


In [ ]:
%cd /content/TripoSR

!pip install -q omegaconf einops transformers trimesh rembg onnxruntime moderngl "imageio[ffmpeg]" gradio huggingface-hub
!pip install -q xatlas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.4/750.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 9.2 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers==4.35.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.17.3 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 0.17.3 which is incompatible.
accelerate 1.15.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.17.3 which is incompatible.
peft 0.21.0 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.17.3 which is incompatible.
gradio-client 2.7.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.17.3 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.17.3 which is incompatible.
sentence-transformers 5.7.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.17.3 which is incompatible.
sentence-transformers 5.7.0 re

In [ ]:
%cd /content/TripoSR

code = '''
import numpy as np
import torch
from skimage import measure

def marching_cubes(vol, thresh):
    v = vol.detach().cpu().numpy()
    verts, faces, _, _ = measure.marching_cubes(v, level=float(thresh))
    verts = torch.from_numpy(verts.copy()).float()
    faces = torch.from_numpy(faces.astype(np.int64))
    return verts, faces
'''
open("torchmcubes.py", "w").write(code)

/content/TripoSR


330

Overenie kompatibility s torchmcubes.py

In [ ]:
import torch, xatlas, torchmcubes, transformers
print(torch.cuda.is_available(), transformers.__version__)

True 4.35.0


/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Nahranie fotografie priamo cez dialogove okno z PC:

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(image_path)

Saving fish_test.png to fish_test.png
fish_test.png


Spustenie samotného TripoSR modelu s danou footgrafiou a generovanie 3D výstupu:

In [ ]:
!python run.py "{image_path}" --output-dir output/

2026-10-04 12:15:03,062 - INFO - Initializing model ...
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-10-04 12:15:18,457 - INFO - Initializing model finished in 15395.56ms.
2026-10-04 12:15:18,458 - INFO - Processing images ...
2026-10-04 12:16:30,155 - INFO - Processing images finished 